# MISOCP Global

Adaptive global MISOCP notebook. It first attempts a single-window solve, then falls back to chunked windows when needed.
Chunked results are treated as near-optimal references rather than global oracles.

> Validation fix baseline recorded before the root-power / reactive-base / tie-breaker repairs:
> `max_root_p_abs_err_kw ~= 148.8`, `max_trafo_loading_abs_err_pct ~= 52.7`.
>
> Current post-fix baseline recorded before the slack / root-q / root-fit diagnostics enhancement:
> `max_root_p_abs_err_kw ~= 48`, `max_trafo_loading_abs_err_pct ~= 52`.
>
> Floor-check / refinement gate will next focus on `p95_soc_slack`, `floor_mean_abs_solver_feeder_gap_kw`, and the resulting `physics_refinement_status`.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from IPython.display import Markdown, display
import numpy as np
import pandas as pd

import configs as configs_pkg
import configs.experiment_config as config_core
import configs.profiles as config_profiles
from controllers import mpc as mpc_pkg
from controllers.mpc import global_socp_mpc as misocp_core
from scripts.builder import build_env
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts import mainline_compare as misocp_nb

config_core = importlib.reload(config_core)
config_profiles = importlib.reload(config_profiles)
configs_pkg = importlib.reload(configs_pkg)
mpc_pkg = importlib.reload(mpc_pkg)
misocp_core = importlib.reload(misocp_core)
grid_nb = importlib.reload(grid_nb)
misocp_nb = importlib.reload(misocp_nb)
compose_experiment_config = config_profiles.compose_experiment_config
print(f"Reloaded config core from: {config_core.__file__}")
print(f"Reloaded config profiles from: {config_profiles.__file__}")
print(f"Reloaded controllers.mpc package from: {getattr(mpc_pkg, '__file__', '<package>')}")
print(f"Reloaded MISOCP solver core from: {misocp_core.__file__}")
print(f"Reloaded grid notebook helpers from: {grid_nb.__file__}")
print(f"Reloaded MISOCP notebook helpers from: {misocp_nb.__file__}")

FullHorizonProblemInput = misocp_core.FullHorizonProblemInput
GlobalMISOCPProblem = misocp_core.GlobalMISOCPProblem
GurobiSolveConfig = misocp_core.GurobiSolveConfig
default_primary_solve_config = misocp_core.default_primary_solve_config
default_retry_solve_config = misocp_core.default_retry_solve_config
apply_notebook_experiment_settings = grid_nb.apply_notebook_experiment_settings
plot_global_misocp_validation = grid_nb.plot_global_misocp_validation
build_chunk_boundary_soc_df = misocp_nb.build_chunk_boundary_soc_df
build_full_horizon_step_df = misocp_nb.build_full_horizon_step_df
build_misocp_validation_artifacts = misocp_nb.build_misocp_validation_artifacts
build_misocp_validation_df = misocp_nb.build_misocp_validation_df
build_root_q_diagnostic_df = misocp_nb.build_root_q_diagnostic_df
build_soc_relaxation_diagnostics = misocp_nb.build_soc_relaxation_diagnostics
format_solver_summary = misocp_nb.format_solver_summary
summarize_misocp_validation = misocp_nb.summarize_misocp_validation
validate_misocp_result_schema = misocp_nb.validate_misocp_result_schema

PROJECT_ROOT = project_root
DATA_DIR = PROJECT_ROOT / "data"
PREDICTION_MODE = "perfect"
TEST_START_DATE = "2020-06-01"
TEST_END_DATE = "2020-06-07"
W_SOC_PEN = 2.0
LOAD_SCALE = [10.0] * 5
PV_SCALE = [5.0] * 5
BATTERY_CAPACITY_KWH = 20.0
BATTERY_MAX_POWER_KW = 10.0
BATTERY_MAX_CHARGE_RATE = BATTERY_MAX_POWER_KW / BATTERY_CAPACITY_KWH
BATTERY_CONTROLS = {"battery_capacity": BATTERY_CAPACITY_KWH, "max_charge_rate": BATTERY_MAX_CHARGE_RATE}
PRIMARY_WINDOW_STEPS = 96*4
FALLBACK_WINDOW_STEPS = 96
PRIMARY_TIME_LIMIT_SEC = 300.0
RETRY_TIME_LIMIT_SEC = 600.0
TARGET_MIP_GAP = 5e-3
RUN_SINGLE_WINDOW_BENCHMARK_AFTER_CHUNKED = False
SCALING_WINDOWS = [
    ("1 day", 96),
    ("2 days", 192),
    ("full test set", None),
]
SHOW_SCALING_GUROBI_LOG = False
SHOW_FULL_SOLVE_GUROBI_LOG = True
SHOW_NOTEBOOK_SOLVER_SUMMARY = True
SHOW_MODEL_SIZE_ESTIMATE = True
EXPORT_DEBUG_ARTIFACTS = True
THROUGHPUT_TIEBREAKER_EUR_PER_KWH = 1e-4
ROOT_TRADE_FORMULATION_NOTE = (
    "Keep binary u_grid[t] to enforce per-step import/export exclusivity. "
    "Removing it would create a faster continuous SOCP, but can admit simultaneous import/export when tariffs permit."
)

solve_succeeded = False
solve_result = None
solve_step_df = pd.DataFrame()
grid_voltage_df = pd.DataFrame()
solve_summary = pd.Series(dtype=object, name="solve_summary")
solve_headline = pd.Series(dtype=object, name="solve_headline")
floor_diagnostic_summary = pd.Series(dtype=object, name="physics_floor_diagnostic")
refinement_summary = pd.Series(dtype=object, name="refinement_summary")
worst_balance_df = pd.DataFrame()
voltage_issue_df = pd.DataFrame()
battery_summary_df = pd.DataFrame()
debug_artifact_series = pd.Series(dtype=object, name="debug_artifacts")
result_schema_error = None
misocp_validation_rollout = None
misocp_validation_df = pd.DataFrame()
misocp_validation_summary = pd.Series(dtype=object, name="misocp_validation_summary")
soc_relaxation_step_df = pd.DataFrame()
soc_relaxation_worst_df = pd.DataFrame()
soc_relaxation_summary = pd.Series(dtype=object, name="soc_relaxation_summary")
root_q_diagnostic_df = pd.DataFrame()
simultaneous_step_df = pd.DataFrame()
simultaneous_agent_df = pd.DataFrame()
chunk_summary_df = pd.DataFrame()
chunk_boundary_soc_df = pd.DataFrame()
benchmark_summary = pd.Series(dtype=object, name="benchmark_summary")
benchmark_result = None
balance_fig = None
voltage_fig = None
net_load_fig = None
misocp_validation_fig = None
root_alignment_fig = None
validation_scatter_fig = None


def _as_float_list(values) -> list[float]:
    array = np.asarray(values, dtype=np.float32).reshape(-1)
    return [float(value) for value in array.tolist()]


def _mpc_value(cfg, field_name: str, default):
    mpc_cfg = getattr(cfg, "mpc", None)
    return getattr(mpc_cfg, field_name, default)


def build_debug_cfg():
    cfg = compose_experiment_config(profile="base", algorithm="MATD3", model_family="mlp", data_dir=DATA_DIR)
    apply_notebook_experiment_settings(
        cfg,
        prediction_mode=PREDICTION_MODE,
        test_start_date=TEST_START_DATE,
        test_end_date=TEST_END_DATE,
        load_scale=LOAD_SCALE,
        pv_scale=PV_SCALE,
        battery_controls=BATTERY_CONTROLS,
    )
    cfg.reward.w_soc_pen = W_SOC_PEN
    return cfg


def build_effective_config_summary(cfg) -> pd.Series:
    return pd.Series(
        {
            "prediction_mode": PREDICTION_MODE,
            "test_start_date": TEST_START_DATE,
            "test_end_date": TEST_END_DATE,
            "agent_profiles": list(cfg.data.agent_profiles),
            "agent_bus_ids": list(cfg.grid.agent_bus_ids),
            "load_scale": _as_float_list(cfg.data.load_scale),
            "pv_scale": _as_float_list(cfg.data.pv_scale),
            "battery_capacity_kwh": _as_float_list(cfg.env.battery_capacity),
            "max_charge_rate": float(cfg.env.max_charge_rate),
            "efficiency": float(cfg.env.efficiency),
            "init_soc": float(cfg.env.init_soc),
            "soc_min": float(cfg.env.soc_min),
            "soc_max": float(cfg.env.soc_max),
            "soc_target": float(cfg.env.soc_target),
            "future_horizon": int(cfg.env.future_horizon),
            "primary_window_steps": int(PRIMARY_WINDOW_STEPS),
            "window_semantics": "future_horizon is env/forecast; PRIMARY_WINDOW_STEPS is the global MISOCP single-window cap.",
            "import_price_markup_eur_per_kwh": float(cfg.reward.import_price_markup_eur_per_kwh),
            "throughput_tiebreaker_eur_per_kwh": THROUGHPUT_TIEBREAKER_EUR_PER_KWH,
            "branch_current_tiebreaker_eur_per_pu_step": float(_mpc_value(cfg, "branch_current_tiebreaker_eur_per_pu_step", 0.0)),
            "physics_refinement_mode": str(_mpc_value(cfg, "physics_refinement_mode", "none")),
            "physics_refinement_slack_ratio": float(_mpc_value(cfg, "physics_refinement_slack_ratio", 2e-2)),
            "physics_refinement_slack_abs_floor_eur": float(_mpc_value(cfg, "physics_refinement_slack_abs_floor_eur", 2.0)),
            "physics_refinement_slack_ratio_schedule": [float(value) for value in list(_mpc_value(cfg, "physics_refinement_slack_ratio_schedule", [2e-2, 5e-2]))],
            "physics_refinement_slack_abs_floor_schedule_eur": [float(value) for value in list(_mpc_value(cfg, "physics_refinement_slack_abs_floor_schedule_eur", [2.0, 5.0]))],
            "physics_refinement_enable_aggressive_third_tier": bool(_mpc_value(cfg, "physics_refinement_enable_aggressive_third_tier", False)),
            "physics_refinement_time_limit_sec": float(_mpc_value(cfg, "physics_refinement_time_limit_sec", 20.0)),
            "physics_refinement_total_time_limit_sec": float(_mpc_value(cfg, "physics_refinement_total_time_limit_sec", 40.0)),
            "physics_refinement_use_full_start": bool(_mpc_value(cfg, "physics_refinement_use_full_start", True)),
        },
        name="effective_notebook_config",
    )


def build_debug_env_problem():
    cfg = build_debug_cfg()
    env = build_env(cfg, mode="test")
    problem = GlobalMISOCPProblem.from_env(
        env,
        cfg,
        throughput_regularization_eur_per_kwh=THROUGHPUT_TIEBREAKER_EUR_PER_KWH,
    )
    return cfg, env, problem


def collect_wholesale_price_seq(env) -> np.ndarray:
    price_chunks = []
    for episode_idx in range(int(env.num_available_episodes)):
        episode = env._dataset.get_episode(int(episode_idx))
        signals = dict(episode.get("signals", {}))
        price_chunks.append(np.asarray(signals["price"], dtype=np.float32).reshape(-1))
    if not price_chunks:
        return np.zeros((0,), dtype=np.float32)
    return np.concatenate(price_chunks, axis=0).astype(np.float32, copy=False)


def display_price_summary(problem: GlobalMISOCPProblem, wholesale_price_seq: np.ndarray, *, label: str):
    wholesale_price_seq = np.asarray(wholesale_price_seq, dtype=np.float32).reshape(-1)
    import_price_seq = problem.apply_import_price_markup(wholesale_price_seq)
    display(pd.Series({
        "price_label": label,
        "import_price_markup_eur_per_kwh": float(problem.import_price_markup_eur_per_kwh),
        "wholesale_price_mean_eur_per_kwh": float(np.mean(wholesale_price_seq)) if wholesale_price_seq.size else np.nan,
        "wholesale_price_min_eur_per_kwh": float(np.min(wholesale_price_seq)) if wholesale_price_seq.size else np.nan,
        "wholesale_price_max_eur_per_kwh": float(np.max(wholesale_price_seq)) if wholesale_price_seq.size else np.nan,
        "import_price_mean_eur_per_kwh": float(np.mean(import_price_seq)) if import_price_seq.size else np.nan,
        "import_price_min_eur_per_kwh": float(np.min(import_price_seq)) if import_price_seq.size else np.nan,
        "import_price_max_eur_per_kwh": float(np.max(import_price_seq)) if import_price_seq.size else np.nan,
    }, name=f"{label}_price_summary"))


def build_primary_solve_config() -> GurobiSolveConfig:
    default_config = default_primary_solve_config()
    return GurobiSolveConfig(
        time_limit_sec=float(PRIMARY_TIME_LIMIT_SEC),
        mip_gap=float(TARGET_MIP_GAP),
        threads=default_config.threads,
        presolve=default_config.presolve,
        cuts=default_config.cuts,
        heuristics=default_config.heuristics,
        mip_focus=default_config.mip_focus,
    )


def build_retry_solve_config(primary_config: GurobiSolveConfig) -> GurobiSolveConfig:
    default_config = default_retry_solve_config(primary_config)
    return GurobiSolveConfig(
        time_limit_sec=float(RETRY_TIME_LIMIT_SEC),
        mip_gap=float(TARGET_MIP_GAP),
        threads=default_config.threads,
        presolve=default_config.presolve,
        cuts=default_config.cuts,
        heuristics=default_config.heuristics,
        mip_focus=default_config.mip_focus,
    )


def misocp_controller_label(solve_mode: str) -> str:
    return "Global MISOCP (chunked, near-optimal)" if str(solve_mode) == "chunked_window" else "Global MISOCP (single_window)"


def build_solve_headline(result, *, total_steps: int, episode_count: int) -> pd.Series:
    summary = format_solver_summary(result, total_steps=total_steps, episode_count=episode_count)
    headline = {
        "solve_mode": str(summary["solve_mode"]),
        "total_runtime_sec": float(summary["total_runtime_sec"]),
        "mip_gap": float(summary["mip_gap"]),
    }
    if str(summary["solve_mode"]) == "chunked_window":
        headline["max_chunk_runtime_sec"] = float(summary["max_chunk_runtime_sec"])
    return pd.Series(headline, name="solve_headline")


def build_floor_diagnostic_summary(result, *, total_steps: int, episode_count: int) -> pd.Series:
    summary = format_solver_summary(result, total_steps=total_steps, episode_count=episode_count)
    gate_passed = bool(
        np.isfinite(float(summary.get("floor_p95_soc_slack", float("nan"))))
        and np.isfinite(float(summary.get("floor_mean_abs_solver_feeder_gap_kw", float("nan"))))
        and float(summary.get("floor_p95_soc_slack", float("nan"))) < 1e-4
        and float(summary.get("floor_mean_abs_solver_feeder_gap_kw", float("nan"))) < 5.0
    )
    return pd.Series(
        {
            "floor_p95_soc_slack": float(summary.get("floor_p95_soc_slack", float("nan"))),
            "floor_mean_abs_solver_feeder_gap_kw": float(summary.get("floor_mean_abs_solver_feeder_gap_kw", float("nan"))),
            "floor_gate_soc_slack_threshold": 1e-4,
            "floor_gate_gap_threshold_kw": 5.0,
            "floor_gate_passed": gate_passed,
        },
        name="physics_floor_diagnostic",
    )


def build_refinement_summary(result, *, total_steps: int, episode_count: int) -> pd.Series:
    summary = format_solver_summary(result, total_steps=total_steps, episode_count=episode_count)
    refinement_warning = ""
    if bool(summary.get("formulation_tightening_required", False)):
        refinement_warning = "Floor solution failed the physical gate; formulation tightening is required."
    elif summary.get("floor_accepted_tier") not in (None, 0):
        refinement_warning = "Floor solution was accepted only after moving to a higher refinement budget tier; economic comparisons need extra care."
    elif bool(summary.get("high_budget_refinement_warn", False)):
        refinement_warning = "Returned solution used a high refinement budget; compare economic conclusions cautiously."
    elif bool(summary.get("negative_floor_delta_warn", False)):
        refinement_warning = "Floor solution improved the primary objective beyond tolerance; Stage 1 may not be fully optimal."
    return pd.Series(
        {
            "physics_refinement_mode": str(summary.get("physics_refinement_mode", "none")),
            "physics_refinement_status": str(summary.get("physics_refinement_status", "not_enabled")),
            "physics_refinement_runtime_sec": float(summary.get("physics_refinement_runtime_sec", 0.0)),
            "initial_physics_refinement_slack_cap_eur": float(summary.get("initial_physics_refinement_slack_cap_eur", float("nan"))),
            "used_physics_refinement_tier": summary.get("used_physics_refinement_tier"),
            "floor_accepted_tier": summary.get("floor_accepted_tier"),
            "total_tiers_configured": int(summary.get("total_tiers_configured", 0)),
            "physics_refinement_attempt_count": int(summary.get("physics_refinement_attempt_count", 0)),
            "physics_refinement_attempt_caps_eur": list(summary.get("physics_refinement_attempt_caps_eur", []) or []),
            "physics_refinement_cap_utilization": float(summary.get("physics_refinement_cap_utilization", float("nan"))),
            "branch_l_gap_ratio_to_floor": float(summary.get("branch_l_gap_ratio_to_floor", float("nan"))),
            "stage1_primary_objective_eur": float(summary.get("stage1_primary_objective_eur", float("nan"))),
            "floor_primary_objective_eur": float(summary.get("floor_primary_objective_eur", float("nan"))),
            "floor_primary_delta_signed_eur": float(summary.get("floor_primary_delta_signed_eur", float("nan"))),
            "floor_primary_delta_positive_eur": float(summary.get("floor_primary_delta_positive_eur", float("nan"))),
            "physics_refinement_slack_cap_eur": float(summary.get("physics_refinement_slack_cap_eur", float("nan"))),
            "stage2_primary_objective_eur": float(summary.get("stage2_primary_objective_eur", float("nan"))),
            "stage2_objective_slack_eur": float(summary.get("stage2_objective_slack_eur", float("nan"))),
            "stage2_branch_l_objective": float(summary.get("stage2_branch_l_objective", float("nan"))),
            "returned_primary_objective_eur": float(summary.get("returned_primary_objective_eur", float("nan"))),
            "returned_primary_delta_abs_eur": float(summary.get("returned_primary_delta_abs_eur", float("nan"))),
            "returned_primary_delta_pct": float(summary.get("returned_primary_delta_pct", float("nan"))),
            "returned_mean_abs_solver_feeder_gap_kw": float(summary.get("returned_mean_abs_solver_feeder_gap_kw", float("nan"))),
            "returned_max_solver_feeder_gap_kw": float(summary.get("returned_max_solver_feeder_gap_kw", float("nan"))),
            "returned_mean_abs_export_gap_ratio": float(summary.get("returned_mean_abs_export_gap_ratio", float("nan"))),
            "high_budget_refinement_warn": bool(summary.get("high_budget_refinement_warn", False)),
            "returned_solution_source": str(summary.get("returned_solution_source", "stage1")),
            "formulation_tightening_required": bool(summary.get("formulation_tightening_required", False)),
            "negative_floor_delta_warn": bool(summary.get("negative_floor_delta_warn", False)),
            "refinement_status_counts": dict(summary.get("refinement_status_counts", {}) or {}),
            "refinement_warning": refinement_warning,
        },
        name="refinement_summary",
    )


def slice_full_input(full_input: FullHorizonProblemInput, horizon_steps: int | None) -> FullHorizonProblemInput:
    total_steps = full_input.horizon_steps if horizon_steps is None else int(min(max(horizon_steps, 1), full_input.horizon_steps))
    episode_offsets = []
    episode_lengths = []
    episode_indices = []
    for offset, length, episode_idx in zip(
        full_input.episode_offsets.tolist(),
        full_input.episode_lengths.tolist(),
        full_input.episode_indices.tolist(),
        strict=False,
    ):
        offset = int(offset)
        length = int(length)
        if offset >= total_steps:
            break
        truncated_length = min(length, total_steps - offset)
        episode_offsets.append(offset)
        episode_lengths.append(int(truncated_length))
        episode_indices.append(int(episode_idx))
    return FullHorizonProblemInput(
        price_seq=np.asarray(full_input.price_seq[:total_steps], dtype=np.float32),
        load_seq=np.asarray(full_input.load_seq[:, :total_steps], dtype=np.float32),
        pv_seq=np.asarray(full_input.pv_seq[:, :total_steps], dtype=np.float32),
        soc_init=np.asarray(full_input.soc_init, dtype=np.float32),
        timestamps=tuple(full_input.timestamps[:total_steps]),
        episode_offsets=np.asarray(episode_offsets, dtype=np.int32),
        episode_lengths=np.asarray(episode_lengths, dtype=np.int32),
        episode_indices=np.asarray(episode_indices, dtype=np.int32),
    )


def run_solve(
    problem: GlobalMISOCPProblem,
    full_input: FullHorizonProblemInput,
    *,
    export_subsidy: float,
    verbose: bool,
    solve_config: GurobiSolveConfig,
    export_debug: bool,
    debug_tag: str,
    partial_mip_start: dict[str, np.ndarray] | None = None,
):
    return problem.solve_full_horizon(
        price_seq=full_input.price_seq,
        load_seq=full_input.load_seq,
        pv_seq=full_input.pv_seq,
        soc_init=full_input.soc_init,
        export_subsidy=export_subsidy,
        timestamps=full_input.timestamps,
        episode_offsets=full_input.episode_offsets,
        episode_lengths=full_input.episode_lengths,
        verbose=verbose,
        solve_config=solve_config,
        export_debug=export_debug,
        debug_tag=debug_tag,
        partial_mip_start=partial_mip_start,
    )


def window_time_bounds(full_input: FullHorizonProblemInput) -> tuple[str, str]:
    if not full_input.timestamps:
        return ("n/a", "n/a")
    return (str(full_input.timestamps[0]), str(full_input.timestamps[-1]))


def display_model_size_estimate(problem: GlobalMISOCPProblem, full_input: FullHorizonProblemInput, *, label: str):
    if not SHOW_MODEL_SIZE_ESTIMATE:
        return
    start_ts, end_ts = window_time_bounds(full_input)
    print(
        f"{label}: steps={full_input.horizon_steps} episodes={int(full_input.episode_lengths.size)} "
        f"from {start_ts} to {end_ts}"
    )
    n_branches = int(problem.n_branches)
    line_branch_indices = getattr(problem.network, "line_branch_indices", None)
    if line_branch_indices is None:
        branch_is_trafo = np.asarray(getattr(problem.network, "branch_is_trafo", np.zeros((n_branches,), dtype=bool)), dtype=bool)
        n_lines = int(np.sum(~branch_is_trafo))
    else:
        n_lines = int(np.asarray(line_branch_indices, dtype=np.int32).size)
    display(pd.Series({
        "horizon_steps": int(full_input.horizon_steps),
        "episode_count": int(full_input.episode_lengths.size),
        "n_agents": int(problem.n_agents),
        "n_buses": int(problem.n_buses),
        "n_branches": n_branches,
        "n_lines": n_lines,
    }, name=f"{label}_problem_scale"))


def display_solver_summary_table(result, *, total_steps: int, episode_count: int, label: str):
    if not SHOW_NOTEBOOK_SOLVER_SUMMARY:
        return
    summary = format_solver_summary(
        result,
        total_steps=total_steps,
        episode_count=episode_count,
    )
    display(summary.rename(f"{label}_solver_summary"))
    solver_note = str(summary.get("solver_note", "")).strip()
    if solver_note:
        print(f"{label}: {solver_note}")


def summarize_result(result, *, total_steps: int):
    solver_summary = format_solver_summary(result, total_steps=total_steps)
    return {
        "status": str(solver_summary["status_label"]),
        "status_code": int(solver_summary["status_code"]),
        "has_solution": bool(solver_summary["has_solution"]),
        "time_limit_feasible": bool(solver_summary["time_limit_feasible"]),
        "solve_mode": str(solver_summary["solve_mode"]),
        "is_near_optimal": bool(solver_summary.get("is_near_optimal", False)),
        "chunk_count": int(solver_summary.get("chunk_count", 0)),
        "T_total": int(solver_summary["horizon_steps"]),
        "solve_time_sec": float(solver_summary["total_runtime_sec"]),
        "max_chunk_runtime_sec": float(solver_summary.get("max_chunk_runtime_sec", np.nan)),
        "mip_gap": float(solver_summary["mip_gap"]),
        "best_bound": float(solver_summary["best_bound"]),
        "objective_value": float(solver_summary["objective_value"]),
        "sol_count": int(solver_summary["sol_count"]),
        "node_count": float(solver_summary["node_count"]),
        "iter_count": float(solver_summary["iter_count"]),
        "bar_iter_count": float(solver_summary["bar_iter_count"]),
        "economics_scope": "agent_only",
        "agent_purchase_cost_eur": float(result.agent_purchase_cost_eur),
        "agent_export_subsidy_eur": float(result.agent_export_subsidy_eur),
        "agent_net_cost_eur": float(result.agent_net_cost_eur),
        "feeder_purchase_cost_eur": float(result.feeder_purchase_cost_eur),
        "feeder_export_subsidy_eur": float(result.feeder_export_subsidy_eur),
        "feeder_net_cost_eur": float(result.feeder_net_cost_eur),
        "throughput_regularization_eur": float(result.throughput_regularization_eur),
        "physical_tiebreaker_eur": float(result.physical_tiebreaker_eur),
        "physical_tiebreaker_weight": float(result.physical_tiebreaker_weight),
        "physics_refinement_mode": str(solver_summary.get("physics_refinement_mode", "none")),
        "physics_refinement_status": str(solver_summary.get("physics_refinement_status", "not_enabled")),
        "physics_refinement_runtime_sec": float(solver_summary.get("physics_refinement_runtime_sec", 0.0)),
        "stage1_primary_objective_eur": float(solver_summary.get("stage1_primary_objective_eur", float("nan"))),
        "stage2_primary_objective_eur": float(solver_summary.get("stage2_primary_objective_eur", float("nan"))),
        "stage2_objective_slack_eur": float(solver_summary.get("stage2_objective_slack_eur", float("nan"))),
        "stage2_branch_l_objective": float(solver_summary.get("stage2_branch_l_objective", float("nan"))),
        "floor_primary_objective_eur": float(solver_summary.get("floor_primary_objective_eur", float("nan"))),
        "floor_primary_delta_signed_eur": float(solver_summary.get("floor_primary_delta_signed_eur", float("nan"))),
        "floor_primary_delta_positive_eur": float(solver_summary.get("floor_primary_delta_positive_eur", float("nan"))),
        "physics_refinement_slack_cap_eur": float(solver_summary.get("physics_refinement_slack_cap_eur", float("nan"))),
        "returned_primary_objective_eur": float(solver_summary.get("returned_primary_objective_eur", float("nan"))),
        "returned_primary_delta_abs_eur": float(solver_summary.get("returned_primary_delta_abs_eur", float("nan"))),
        "returned_primary_delta_pct": float(solver_summary.get("returned_primary_delta_pct", float("nan"))),
        "returned_solution_source": str(solver_summary.get("returned_solution_source", "stage1")),
        "formulation_tightening_required": bool(solver_summary.get("formulation_tightening_required", False)),
        "negative_floor_delta_warn": bool(solver_summary.get("negative_floor_delta_warn", False)),
        "floor_p95_soc_slack": float(solver_summary.get("floor_p95_soc_slack", float("nan"))),
        "floor_mean_abs_solver_feeder_gap_kw": float(solver_summary.get("floor_mean_abs_solver_feeder_gap_kw", float("nan"))),
        "num_vars": int(result.model_size.num_vars),
        "num_binary_vars": int(result.model_size.num_binary_vars),
        "num_linear_constraints": int(result.model_size.num_linear_constraints),
        "num_quadratic_constraints": int(result.model_size.num_quadratic_constraints),
        "simultaneous_agent_steps": int(result.simultaneous_agent_steps),
        "simultaneous_step_ratio": float(result.simultaneous_step_ratio),
        "max_simultaneous_kw": float(result.max_simultaneous_kw),
    }

cfg_preview = build_debug_cfg()
display(build_effective_config_summary(cfg_preview))
display(pd.Series({
    "primary_window_steps": PRIMARY_WINDOW_STEPS,
    "fallback_window_steps": FALLBACK_WINDOW_STEPS,
    "primary_time_limit_sec": PRIMARY_TIME_LIMIT_SEC,
    "retry_time_limit_sec": RETRY_TIME_LIMIT_SEC,
    "target_mip_gap": TARGET_MIP_GAP,
    "run_single_window_benchmark_after_chunked": RUN_SINGLE_WINDOW_BENCHMARK_AFTER_CHUNKED,
    "show_notebook_solver_summary": SHOW_NOTEBOOK_SOLVER_SUMMARY,
    "show_model_size_estimate": SHOW_MODEL_SIZE_ESTIMATE,
    "physics_refinement_mode": str(_mpc_value(cfg_preview, "physics_refinement_mode", "none")),
    "physics_refinement_slack_ratio": float(_mpc_value(cfg_preview, "physics_refinement_slack_ratio", 2e-2)),
    "physics_refinement_slack_abs_floor_eur": float(_mpc_value(cfg_preview, "physics_refinement_slack_abs_floor_eur", 2.0)),
    "physics_refinement_slack_ratio_schedule": [float(value) for value in list(_mpc_value(cfg_preview, "physics_refinement_slack_ratio_schedule", [2e-2, 5e-2]))],
    "physics_refinement_slack_abs_floor_schedule_eur": [float(value) for value in list(_mpc_value(cfg_preview, "physics_refinement_slack_abs_floor_schedule_eur", [2.0, 5.0]))],
    "physics_refinement_enable_aggressive_third_tier": bool(_mpc_value(cfg_preview, "physics_refinement_enable_aggressive_third_tier", False)),
    "physics_refinement_time_limit_sec": float(_mpc_value(cfg_preview, "physics_refinement_time_limit_sec", 20.0)),
    "physics_refinement_total_time_limit_sec": float(_mpc_value(cfg_preview, "physics_refinement_total_time_limit_sec", 40.0)),
    "physics_refinement_use_full_start": bool(_mpc_value(cfg_preview, "physics_refinement_use_full_start", True)),
    "root_trade_formulation": ROOT_TRADE_FORMULATION_NOTE,
    "debug_dir": str(PROJECT_ROOT / "artifacts" / "misocp_debug"),
}, name="misocp_notebook_runtime"))


In [ ]:
cfg, env, problem = build_debug_env_problem()
try:
    full_input = problem.build_full_horizon_input(env)
    wholesale_price_seq = collect_wholesale_price_seq(env)
    export_subsidy = float(cfg.reward.export_subsidy_eur_per_kwh)
    primary_solve_config = build_primary_solve_config()
    print(f"Prepared full-horizon input with T_total={full_input.horizon_steps} across {len(full_input.episode_lengths)} episode(s).")
    print(ROOT_TRADE_FORMULATION_NOTE)
    display_price_summary(problem, wholesale_price_seq, label="scaling_input")
    display(pd.Series({
        "validation_root_power_source": "trafo_p_signed_kw",
        "agent_bus_q_base_nonzero_count": int(np.count_nonzero(np.abs(np.asarray(problem.network.q_base_mvar, dtype=np.float32)[np.asarray(problem.network.agent_bus_positions, dtype=np.int32)]) > 1e-9)),
    }, name="misocp_physics_alignment"))
    scaling_rows = []
    for window_label, window_steps in SCALING_WINDOWS:
        trial_input = slice_full_input(full_input, window_steps)
        print(f"Starting {window_label} solve (T={trial_input.horizon_steps})...")
        display_model_size_estimate(problem, trial_input, label=window_label.replace(" ", "_"))
        trial_result = run_solve(
            problem,
            trial_input,
            export_subsidy=export_subsidy,
            verbose=SHOW_SCALING_GUROBI_LOG,
            solve_config=primary_solve_config,
            export_debug=EXPORT_DEBUG_ARTIFACTS,
            debug_tag=f"scaling_{window_label.replace(' ', '_')}",
        )
        display_solver_summary_table(
            trial_result,
            total_steps=trial_input.horizon_steps,
            episode_count=int(trial_input.episode_lengths.size),
            label=window_label.replace(" ", "_"),
        )
        row = summarize_result(trial_result, total_steps=trial_input.horizon_steps)
        row["window"] = window_label
        scaling_rows.append(row)
        print(
            f"Done: {window_label} status={row['status']} runtime={row['solve_time_sec']:.2f}s "
            f"gap={row['mip_gap']:.4f} binaries={row['num_binary_vars']} nodes={row['node_count']:.0f}"
        )
    display(pd.DataFrame(scaling_rows))
finally:
    env.close()


In [ ]:
cfg, env, problem = build_debug_env_problem()
solve_succeeded = False
solve_result = None
solve_step_df = pd.DataFrame()
grid_voltage_df = pd.DataFrame()
solve_summary = pd.Series(dtype=object, name="solve_summary")
solve_headline = pd.Series(dtype=object, name="solve_headline")
floor_diagnostic_summary = pd.Series(dtype=object, name="physics_floor_diagnostic")
refinement_summary = pd.Series(dtype=object, name="refinement_summary")
worst_balance_df = pd.DataFrame()
voltage_issue_df = pd.DataFrame()
battery_summary_df = pd.DataFrame()
debug_artifact_series = pd.Series(dtype=object, name="debug_artifacts")
misocp_validation_rollout = None
misocp_validation_df = pd.DataFrame()
misocp_validation_summary = pd.Series(dtype=object, name="misocp_validation_summary")
simultaneous_step_df = pd.DataFrame()
simultaneous_agent_df = pd.DataFrame()
chunk_summary_df = pd.DataFrame()
chunk_boundary_soc_df = pd.DataFrame()
benchmark_summary = pd.Series(dtype=object, name="benchmark_summary")
benchmark_result = None
try:
    full_input = problem.build_full_horizon_input(env)
    wholesale_price_seq = collect_wholesale_price_seq(env)
    export_subsidy = float(cfg.reward.export_subsidy_eur_per_kwh)
    primary_solve_config = build_primary_solve_config()
    retry_solve_config = build_retry_solve_config(primary_solve_config)
    resolved_agent_profiles = list(cfg.data.agent_profiles)
    resolved_agent_bus_ids = list(cfg.grid.agent_bus_ids)
    display(build_effective_config_summary(cfg))
    display(pd.Series({
        "primary_window_steps": PRIMARY_WINDOW_STEPS,
        "fallback_window_steps": FALLBACK_WINDOW_STEPS,
        "primary_time_limit_sec": PRIMARY_TIME_LIMIT_SEC,
        "retry_time_limit_sec": RETRY_TIME_LIMIT_SEC,
        "target_mip_gap": TARGET_MIP_GAP,
        "run_single_window_benchmark_after_chunked": RUN_SINGLE_WINDOW_BENCHMARK_AFTER_CHUNKED,
        "branch_current_tiebreaker_eur_per_pu_step": float(_mpc_value(cfg, "branch_current_tiebreaker_eur_per_pu_step", 0.0)),
        "physics_refinement_mode": str(_mpc_value(cfg, "physics_refinement_mode", "none")),
        "physics_refinement_slack_ratio": float(_mpc_value(cfg, "physics_refinement_slack_ratio", 2e-2)),
        "physics_refinement_slack_abs_floor_eur": float(_mpc_value(cfg, "physics_refinement_slack_abs_floor_eur", 2.0)),
        "physics_refinement_slack_ratio_schedule": [float(value) for value in list(_mpc_value(cfg, "physics_refinement_slack_ratio_schedule", [2e-2, 5e-2]))],
        "physics_refinement_slack_abs_floor_schedule_eur": [float(value) for value in list(_mpc_value(cfg, "physics_refinement_slack_abs_floor_schedule_eur", [2.0, 5.0]))],
        "physics_refinement_enable_aggressive_third_tier": bool(_mpc_value(cfg, "physics_refinement_enable_aggressive_third_tier", False)),
        "physics_refinement_time_limit_sec": float(_mpc_value(cfg, "physics_refinement_time_limit_sec", 20.0)),
        "physics_refinement_total_time_limit_sec": float(_mpc_value(cfg, "physics_refinement_total_time_limit_sec", 40.0)),
        "physics_refinement_use_full_start": bool(_mpc_value(cfg, "physics_refinement_use_full_start", True)),
    }, name="adaptive_solve_config"))
    display(pd.Series({
        "validation_root_power_source": "trafo_p_signed_kw",
        "agent_bus_q_base_nonzero_count": int(np.count_nonzero(np.abs(np.asarray(problem.network.q_base_mvar, dtype=np.float32)[np.asarray(problem.network.agent_bus_positions, dtype=np.int32)]) > 1e-9)),
    }, name="misocp_physics_alignment"))
    print(f"Attempting adaptive global MISOCP solve for T_total={full_input.horizon_steps} steps...")
    display_price_summary(problem, wholesale_price_seq, label="adaptive_full_horizon_input")
    display_model_size_estimate(problem, full_input, label="adaptive_full_horizon")
    solve_result = problem.solve_adaptive_full_horizon(
        full_input,
        export_subsidy=export_subsidy,
        verbose=SHOW_FULL_SOLVE_GUROBI_LOG,
        primary_window_steps=PRIMARY_WINDOW_STEPS,
        fallback_window_steps=FALLBACK_WINDOW_STEPS,
        solve_config=primary_solve_config,
        retry_solve_config=retry_solve_config,
        export_debug=EXPORT_DEBUG_ARTIFACTS,
        debug_tag="adaptive_global_misocp",
    )
    display_solver_summary_table(
        solve_result,
        total_steps=full_input.horizon_steps,
        episode_count=int(full_input.episode_lengths.size),
        label=f"adaptive_{solve_result.solve_mode}",
    )
    solve_headline = build_solve_headline(
        solve_result,
        total_steps=full_input.horizon_steps,
        episode_count=int(full_input.episode_lengths.size),
    )
    floor_diagnostic_summary = build_floor_diagnostic_summary(
        solve_result,
        total_steps=full_input.horizon_steps,
        episode_count=int(full_input.episode_lengths.size),
    )
    refinement_summary = build_refinement_summary(
        solve_result,
        total_steps=full_input.horizon_steps,
        episode_count=int(full_input.episode_lengths.size),
    )
    chunk_summary_df = pd.DataFrame(list(solve_result.chunk_summaries or []))
    summary_dict = summarize_result(solve_result, total_steps=full_input.horizon_steps)
    summary_dict["num_episodes"] = int(full_input.episode_lengths.size)
    summary_dict["controller_label"] = misocp_controller_label(solve_result.solve_mode)
    solve_summary = pd.Series(summary_dict, name="solve_summary")
    result_schema_ok = True
    if solve_result.has_solution:
        try:
            validate_misocp_result_schema(solve_result)
        except ValueError as exc:
            result_schema_ok = False
            result_schema_error = str(exc)
            display(Markdown(f"**MISOCP result schema mismatch**  \n{result_schema_error}"))
    if solve_result.has_solution and result_schema_ok:
        solve_step_df = build_full_horizon_step_df(problem, full_input, solve_result)
        if solve_result.solve_mode == "chunked_window":
            chunk_boundary_soc_df = build_chunk_boundary_soc_df(problem, full_input, solve_result)
        controller_label = misocp_controller_label(solve_result.solve_mode)
        validation_artifacts = build_misocp_validation_artifacts(
            env,
            problem,
            full_input,
            solve_result,
            controller_label=controller_label,
            export_subsidy=export_subsidy,
            agent_profiles=resolved_agent_profiles,
            agent_bus_ids=resolved_agent_bus_ids,
            v_min_pu=float(cfg.grid.v_min_pu),
            v_max_pu=float(cfg.grid.v_max_pu),
        )
        misocp_validation_rollout = validation_artifacts["rollout"]
        misocp_validation_df = validation_artifacts["validation_df"]
        misocp_validation_summary = validation_artifacts["validation_summary"]
        soc_relaxation_step_df = validation_artifacts["soc_relaxation_step_df"]
        soc_relaxation_worst_df = validation_artifacts["soc_relaxation_worst_df"]
        soc_relaxation_summary = validation_artifacts["soc_relaxation_summary"]
        root_q_diagnostic_df = validation_artifacts["root_q_diagnostic_df"]
        replay_step_df = validation_artifacts["step_df"]
        if not replay_step_df.empty and "pp_root_p_kw" in replay_step_df.columns:
            solve_step_df = solve_step_df.merge(
                replay_step_df.loc[:, [
                    "global_step",
                    "pp_root_p_kw",
                    "solver_feeder_gap_kw",
                    "replay_feeder_gap_kw",
                    "background_q_base_total_kvar",
                    "network_q_loss_proxy_kvar",
                    "root_q_residual_kvar",
                    "soc_slack_max",
                    "soc_slack_mean",
                    "soc_slack_p95_global",
                ]].drop_duplicates(subset=["global_step"]),
                on="global_step",
                how="left",
            )
        if RUN_SINGLE_WINDOW_BENCHMARK_AFTER_CHUNKED and solve_result.solve_mode == "chunked_window":
            print("Running optional single-window benchmark with partial MIP start from the chunked solution...")
            benchmark_solve_config = GurobiSolveConfig(
                time_limit_sec=float(RETRY_TIME_LIMIT_SEC),
                mip_gap=float(TARGET_MIP_GAP),
                threads=primary_solve_config.threads,
                presolve=primary_solve_config.presolve,
                cuts=primary_solve_config.cuts,
                heuristics=primary_solve_config.heuristics,
                mip_focus=primary_solve_config.mip_focus,
            )
            benchmark_result = run_solve(
                problem,
                full_input,
                export_subsidy=export_subsidy,
                verbose=SHOW_FULL_SOLVE_GUROBI_LOG,
                solve_config=benchmark_solve_config,
                export_debug=EXPORT_DEBUG_ARTIFACTS,
                debug_tag="adaptive_global_misocp_benchmark",
                partial_mip_start=problem.build_partial_mip_start(solve_result),
            )
            benchmark_summary = format_solver_summary(
                benchmark_result,
                total_steps=full_input.horizon_steps,
                episode_count=int(full_input.episode_lengths.size),
            )
        solve_succeeded = True
    elif solve_result.has_solution:
        print("Skipping notebook diagnostics because solve_result uses an outdated schema. Re-run the first import cell and the solve cell, or restart the kernel.")
    else:
        print(f"Adaptive global MISOCP ended without an incumbent. Final status={solve_result.status_label!r}")
    display(solve_headline)
    display(solve_summary)
    display(floor_diagnostic_summary)
    display(refinement_summary)
    if not chunk_summary_df.empty:
        display(chunk_summary_df)
    if solve_succeeded:
        if not chunk_boundary_soc_df.empty:
            display(chunk_boundary_soc_df)
        display(misocp_validation_summary)
        display(Markdown("Fit interpretation guide: `k鈮?, b鈮?, R虏>0.99` => consistent; `k鈮?1` => sign error; `k鈮?000/0.001` => unit mismatch; `R虏<0.9` => nonlinear/noise or model-difference dominated."))
        display(soc_relaxation_summary)
        if misocp_validation_df.empty:
            print("misocp_validation_df is empty: validation replay produced no rows.")
        else:
            display(misocp_validation_df)
        if not soc_relaxation_worst_df.empty:
            display(soc_relaxation_worst_df)
        if not root_q_diagnostic_df.empty:
            display(root_q_diagnostic_df)
        if not debug_artifact_series.empty:
            display(debug_artifact_series)
        display(solve_step_df.head())
        display(solve_step_df.tail())
        if benchmark_result is not None:
            display(benchmark_summary.rename("single_window_benchmark_summary"))
    else:
        debug_artifact_series = pd.Series(dict(getattr(solve_result, "debug_artifacts", {})), name="debug_artifacts")
        if not debug_artifact_series.empty:
            display(debug_artifact_series)
finally:
    env.close()


## Simultaneous Charge/Discharge Note

The current MISOCP formulation keeps `p_charge >= 0` and `p_discharge >= 0` as independent continuous variables. It limits both with local power bounds and energy balance, and it adds a tiny throughput regularization term, but it does not impose a hard mutual-exclusion constraint on battery charging and discharging.

So if you see simultaneous charge/discharge in the tables below, treat it as a formulation-permitted behavior to inspect, not as a plotting artifact.


In [ ]:
misocp_validation_fig = None
if not solve_succeeded:
    print("skip plotting because solve has no incumbent")
else:
    if not solve_step_df.empty:
        display(solve_step_df.loc[:, [
            "timestamp",
            "root_net_exchange_kw",
            "balance_residual_kw",
            "max_line_loading_pct",
            "trafo_loading_pct",
        ]].head(20))
    if misocp_validation_rollout is not None:
        misocp_validation_fig = plot_global_misocp_validation(misocp_validation_rollout)
        display(misocp_validation_fig)
